# Embryo Segmentation -- Atlas CS17 GPU Training + Held-Out Slice Evaluation (one of 4 group models)

This notebook trains **one model for one group** of the balanced groups built by the data-prep
notebook (default: 4 groups, ~22 structures each, 87 in total). Change the single `GROUP_INDEX`
variable in Section 0 (0, 1, 2, 3) and re-run top to bottom to train the next group's model. Each
group downloads its own dataset, trains its own model and pushes its own checkpoints to its own
folder, without touching any other group's results.

It reads from and writes to **new Hugging Face repos only**:

* dataset (input):     `senku111/atlas-cs17-4groups-dataset`
* checkpoints (output): `senku111/atlas-cs17-4groups-nnunet-training`

Configuration cell asserts these are not the old repos, so nothing can merge with earlier runs.

Steps:
1. resolve `GROUP_INDEX` against `all_groups_manifest.json` to find that group's dataset, and
   download only that group's frozen dataset;
2. plan/preprocess with nnU-Net;
3. train fold 0 (250 epochs), uploading checkpoints to HF every 30 minutes and after the fold;
4. restore that group's checkpoints automatically if a GPU session was interrupted;
5. evaluate on the group's **held-out slices** (per-class Dice + overlay visualisations).

VHE is not used anywhere in this notebook (it has no ground truth, so it cannot be scored).


In [ ]:
# ============================================================
# PARAMETERS
# ------------------------------------------------------------
# This cell is tagged `parameters`: the pipeline runner / web form
# injects values for every variable below (papermill), and everything
# after this cell derives its paths and settings from them.
# Running the notebook by hand uses these defaults, which reproduce
# the original CS17 run.
# ============================================================

# Local scratch root. Kaggle: "/kaggle/working"; Colab: "/content".
WORK_DIR = "/kaggle/working"

# Single shared "training checkpoints" repo (repo_type=model). Each group's model
# is pushed to its own folder inside it, so training group 3 never overwrites
# group 0's checkpoints.
HF_TRAINING_REPO_ID = "senku111/atlas-cs17-4groups-nnunet-training"
# The dataset repo the data-preparation notebook pushed to.
DATASET_HF_REPO_ID = "senku111/atlas-cs17-4groups-dataset"
# Must match the SPECIMEN_ID used by the data-preparation notebook.
SPECIMEN_ID = "CS17-6521"

# *** THE ONE VARIABLE TO CHANGE TO TRAIN A DIFFERENT GROUP ***
# Set GROUP_INDEX to the group this run trains (0, 1, 2, ...), then run the
# notebook top to bottom. DATASET_ID / DATASET_NAME / RESULTS_PATH_IN_REPO are
# resolved from that group's entry of all_groups_manifest.json below.
GROUP_INDEX = 0

N_FOLDS = 1
CONFIGURATION = "2d"
TRAINER = "nnUNetTrainer_250epochs"
# Upload the current training state to HF every N minutes.
CHECKPOINT_UPLOAD_MINUTES = 30
# Resume from checkpoint_latest.pth when a fold already exists; False stops instead.
RESUME_FROM_CHECKPOINT = True
# Must stay False. Delete results manually only to intentionally restart a run.
CLEAN_RESULTS = False

# ============================================================ # end parameters # ====

In [1]:
# ============================================================
# 0. CONFIGURATION
# ============================================================
import os, json, subprocess, sys, shutil, time, threading, signal
from pathlib import Path

# Single shared "training checkpoints" repo. Each group's model is pushed to its OWN folder
# inside this repo (RESULTS_PATH_IN_REPO, resolved in Section 1 below), so training group 3
# never overwrites group 0's checkpoints.

# --- safety: never read from / write to the old repos ---
OLD_REPOS = ["senku111/3d-module-new", "senku111/atlas-cs17-nnunet", "senku111/atlas-cs17-nnunet-training-v3-new"]
assert HF_TRAINING_REPO_ID not in OLD_REPOS and DATASET_HF_REPO_ID not in OLD_REPOS, \
    "A repo id points at an OLD repo -- use the new repos."

# *** Must exactly match the SPECIMEN_ID the data-prep notebook used (STAGE + ATLAS_SPECIMEN_ID) ***
# so this notebook looks up the same specimen's all_groups_manifest.json on Hugging Face.
if not SPECIMEN_ID:
    raise RuntimeError(
        "SPECIMEN_ID is not set. Copy the exact SPECIMEN_ID value the data-prep notebook used "
        "for CS17 (STAGE + ATLAS_SPECIMEN_ID) and set it here."
    )

# ------------------------------------------------------------
# *** THE ONE VARIABLE TO CHANGE TO TRAIN A DIFFERENT MODEL ***
# ------------------------------------------------------------
# The data-prep notebook split the embryo's 87 structures into 4 balanced groups,
# each its own nnU-Net dataset (listed in all_groups_manifest.json). Set GROUP_INDEX to the
# group you want THIS run to train, then re-run the notebook top to bottom: it will download
# that group's dataset, train that group's model, and push it to that group's own folder,
# without touching any other group's results.
#
# DATASET_ID / DATASET_NAME are resolved automatically from the manifest in Section 1 below —
# do not hardcode them here.

DATASET_ID = None    # resolved in Section 1
DATASET_NAME = None  # resolved in Section 1


# Upload the current training state to HF every 30 minutes.

# If True, every fold is allowed to resume from checkpoint_latest.pth.
# If False, an existing fold is left untouched and the notebook stops
# rather than accidentally restarting it.

# Safety guard: this must stay False. Delete results manually only if
# you intentionally want to blow away an existing run and start over.
if CLEAN_RESULTS:
    raise RuntimeError(
        "CLEAN_RESULTS is intentionally disabled in this notebook. "
        "Delete results manually only if you intentionally want a new run."
    )

# Local working dirs are namespaced by GROUP_INDEX so switching groups and rerunning in the
# same Kaggle session never mixes one group's raw/preprocessed data with another's.
BASE_DIR = f"{WORK_DIR}/embryo_segmentation_training_group{GROUP_INDEX:02d}"
HF_DIR = os.path.join(BASE_DIR, "hf_dataset")
NNUNET_RAW = os.path.join(BASE_DIR, "nnUNet_raw")
NNUNET_PREPROCESSED = os.path.join(BASE_DIR, "nnUNet_preprocessed")
NNUNET_RESULTS = os.path.join(BASE_DIR, "nnUNet_results")

# Separate persistent local cache for checkpoint packaging (staging copies,
# so an HF upload never reads a file nnU-Net is writing to at that instant).
CHECKPOINT_CACHE = os.path.join(BASE_DIR, "checkpoint_upload_cache")

for d in [HF_DIR, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS, CHECKPOINT_CACHE]:
    os.makedirs(d, exist_ok=True)

os.environ["nnUNet_raw"] = NNUNET_RAW
os.environ["nnUNet_preprocessed"] = NNUNET_PREPROCESSED
os.environ["nnUNet_results"] = NNUNET_RESULTS

# Where THIS group's checkpoints/results live inside HF_TRAINING_REPO_ID.
# Set once DATASET_NAME is resolved in Section 1 (do not hardcode).
RESULTS_PATH_IN_REPO = None

print("Training HF dataset repo:", DATASET_HF_REPO_ID)
print("Training checkpoint repo:", HF_TRAINING_REPO_ID, "(repo_type=model)")
print("GROUP_INDEX:", GROUP_INDEX, "  <-- change this to train a different group's model")
print("Trainer:", TRAINER)
print("Folds:", N_FOLDS)
print("Checkpoint interval:", CHECKPOINT_UPLOAD_MINUTES, "minutes")


Training HF dataset repo: senku111/atlas-cs17-4groups-dataset
Training checkpoint repo: senku111/atlas-cs17-4groups-nnunet-training (repo_type=model)
GROUP_INDEX: 1   <-- change this to train a different group's model
Trainer: nnUNetTrainer_250epochs
Folds: 1
Checkpoint interval: 15 minutes


## 1. Resolve `GROUP_INDEX` against the groups manifest, then download that group's dataset

The data-prep notebook pushed one `all_groups_manifest.json` listing every group it built. This
section downloads just that manifest first, resolves `GROUP_INDEX` into a concrete
`DATASET_ID`/`DATASET_NAME`/folder, and only then downloads that one group's dataset — not every
other group's data.


In [2]:
# ============================================================
# 1. RESOLVE GROUP_INDEX, THEN DOWNLOAD ONLY THAT GROUP'S DATASET
# ============================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "nnunetv2"], check=True)
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        raise RuntimeError("HF_TOKEN unavailable.") from e

# --- Step 1: fetch just the manifest listing every group the data-prep notebook built ---
manifest_dir = os.path.join(BASE_DIR, "manifest_only")
if os.path.exists(manifest_dir):
    shutil.rmtree(manifest_dir)
os.makedirs(manifest_dir, exist_ok=True)

snapshot_download(
    repo_id=DATASET_HF_REPO_ID, repo_type="dataset", token=HF_TOKEN,
    allow_patterns=[f"training_dataset/{SPECIMEN_ID}/reports/all_groups_manifest.json"],
    local_dir=manifest_dir,
)
manifest_path = os.path.join(manifest_dir, "training_dataset", SPECIMEN_ID, "reports", "all_groups_manifest.json")
assert os.path.exists(manifest_path), (
    f"Missing {manifest_path} -- was the data-prep notebook's grouped-export section run and "
    f"pushed to {DATASET_HF_REPO_ID}?"
)
with open(manifest_path) as f:
    all_groups_manifest = json.load(f)

group_entries = {g["group_index"]: g for g in all_groups_manifest["groups"]}
assert GROUP_INDEX in group_entries, (
    f"GROUP_INDEX={GROUP_INDEX} not found in the manifest. Available groups: {sorted(group_entries)}"
)
this_group = group_entries[GROUP_INDEX]
DATASET_ID = this_group["dataset_id"]
DATASET_NAME = this_group["dataset_name"]
RESULTS_PATH_IN_REPO = f"group{GROUP_INDEX:02d}_{DATASET_NAME}/nnUNet_results"

print(f"GROUP_INDEX {GROUP_INDEX} -> Dataset{DATASET_ID:03d}_{DATASET_NAME} "
      f"({this_group['n_classes']} classes)")
print("Atlas IDs in this group:", this_group["atlas_ids"])
print("This group's results will be pushed to:", f"{HF_TRAINING_REPO_ID}/{RESULTS_PATH_IN_REPO}")

# --- Step 2: download only this group's dataset folder ---
if os.path.exists(HF_DIR):
    shutil.rmtree(HF_DIR)
os.makedirs(HF_DIR, exist_ok=True)

snapshot_download(
    repo_id=DATASET_HF_REPO_ID, repo_type="dataset", token=HF_TOKEN,
    allow_patterns=[f"training_dataset/{SPECIMEN_ID}/{this_group['dataset_folder']}/*"],
    local_dir=HF_DIR,
)
print("Downloaded training dataset snapshot to", HF_DIR)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 4.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.3 which is incompatible.


Fetching ... files: 0it [00:00, ?it/s]

GROUP_INDEX 1 -> Dataset702_AtlasCS17FourGroup01 (22 classes)
Atlas IDs in this group: [3, 4, 5, 6, 9, 11, 18, 19, 23, 29, 32, 34, 35, 37, 39, 42, 48, 59, 60, 61, 71, 79]
This group's results will be pushed to: senku111/atlas-cs17-4groups-nnunet-training/group01_AtlasCS17FourGroup01/nnUNet_results


Fetching ... files: 0it [00:00, ?it/s]

Downloaded training dataset snapshot to /kaggle/working/embryo_segmentation_training_group01/hf_dataset


## 1B. Verify this group's frozen training dataset + load its held-out split


In [3]:
# ============================================================
# 1B. VERIFY FROZEN TRAINING DATASET
# ============================================================
dataset_dir = os.path.join(HF_DIR, "training_dataset", SPECIMEN_ID, f"Dataset{DATASET_ID:03d}_{DATASET_NAME}")

dataset_json_path = os.path.join(dataset_dir, "dataset.json")
split_manifest_path = os.path.join(dataset_dir, "split_manifest.json")
images_dir = os.path.join(dataset_dir, "imagesTr")
labels_dir = os.path.join(dataset_dir, "labelsTr")
heldout_images_dir = os.path.join(dataset_dir, "heldout", "images")
heldout_labels_dir = os.path.join(dataset_dir, "heldout", "labels")

assert os.path.exists(dataset_json_path), f"Missing: {dataset_json_path}"
assert os.path.exists(split_manifest_path), f"Missing: {split_manifest_path} — was this group's " \
    "dataset built by the current (grouped) data-prep notebook, which writes this per group?"
assert os.path.isdir(images_dir) and os.path.isdir(labels_dir)
assert os.path.isdir(heldout_images_dir) and os.path.isdir(heldout_labels_dir), \
    "Missing heldout/images or heldout/labels — this dataset predates the leakage fix; rebuild it."

with open(dataset_json_path) as f:
    dataset_json = json.load(f)
with open(split_manifest_path) as f:
    split_manifest = json.load(f)

N_FOREGROUND_CLASSES = len(dataset_json["labels"]) - 1
print(f"dataset.json: {N_FOREGROUND_CLASSES} foreground classes, numTraining={dataset_json['numTraining']}")
print(f"split_manifest: {split_manifest['n_train']} train slices, {split_manifest['n_holdout']} held-out slices "
      f"(seed={split_manifest['seed']})")
print("SCOPE NOTE:", split_manifest["note"])

# --- leakage guard: verify on disk, not just trust the manifest's claim ---
tr_file_stems = {f.replace("_0000.png", "") for f in os.listdir(images_dir) if f.endswith(".png")}
heldout_stems_in_tr = tr_file_stems & set(split_manifest["holdout_cases"])
assert not heldout_stems_in_tr, \
    f"[DATA LEAK] {len(heldout_stems_in_tr)} held-out case(s) found inside imagesTr: {heldout_stems_in_tr}"
assert dataset_json["numTraining"] == len(tr_file_stems) == split_manifest["n_train"], \
    f"[DATA PROBLEM] numTraining ({dataset_json['numTraining']}) doesn't match imagesTr file count " \
    f"({len(tr_file_stems)}) or split_manifest's n_train ({split_manifest['n_train']})"
print(f"\nVerified: 0 of {split_manifest['n_holdout']} held-out cases present in imagesTr/labelsTr, "
      f"and numTraining matches the actual file count exactly. Held-out slices were never indexed "
      f"by nnU-Net's own planning, preprocessing, or cross-validation.")


dataset.json: 22 foreground classes, numTraining=615
split_manifest: 615 train slices, 153 held-out slices (seed=42)
SCOPE NOTE: Held-out cases are held-out SLICES from this one specimen, not a held-out specimen, and are stored in heldout/images + heldout/labels, physically separate from imagesTr/labelsTr, so nnU-Net's own planning/preprocessing/cross-validation never sees them.

Verified: 0 of 153 held-out cases present in imagesTr/labelsTr, and numTraining matches the actual file count exactly. Held-out slices were never indexed by nnU-Net's own planning, preprocessing, or cross-validation.


## 1C. Preprocessing state

In [4]:
# ============================================================
# 1C. TRAINING STATE / PREPROCESSING
# ============================================================
print("This notebook will run nnU-Net planning and preprocessing locally.")
print("Preprocessing is deterministic and can be regenerated from the frozen HF dataset.")

preprocessed_dataset_dir = os.path.join(NNUNET_PREPROCESSED, f"Dataset{DATASET_ID:03d}_{DATASET_NAME}")
print("Expected preprocessed directory:", preprocessed_dataset_dir)

raw_dataset_dir = os.path.join(NNUNET_RAW, f"Dataset{DATASET_ID:03d}_{DATASET_NAME}")
if not os.path.exists(raw_dataset_dir):
    shutil.copytree(dataset_dir, raw_dataset_dir)
    print("Copied dataset into nnUNet_raw layout at", raw_dataset_dir)


This notebook will run nnU-Net planning and preprocessing locally.
Preprocessing is deterministic and can be regenerated from the frozen HF dataset.
Expected preprocessed directory: /kaggle/working/embryo_segmentation_training_group01/nnUNet_preprocessed/Dataset702_AtlasCS17FourGroup01
Copied dataset into nnUNet_raw layout at /kaggle/working/embryo_segmentation_training_group01/nnUNet_raw/Dataset702_AtlasCS17FourGroup01


## 2. GPU + nnU-Net planning and preprocessing

In [5]:
# ============================================================
# 2. PLAN AND PREPROCESS WITH nnU-Net
# ============================================================
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Stop rather than training on CPU.")

print("GPU:", torch.cuda.get_device_name(0))

for key in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    print(key, "=", os.environ[key])

subprocess.run([
    "nnUNetv2_plan_and_preprocess", "-d", str(DATASET_ID), "--verify_dataset_integrity"
], check=True)


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
nnUNet_raw = /kaggle/working/embryo_segmentation_training_group01/nnUNet_raw
nnUNet_preprocessed = /kaggle/working/embryo_segmentation_training_group01/nnUNet_preprocessed
nnUNet_results = /kaggle/working/embryo_segmentation_training_group01/nnUNet_results
Fingerprint extraction...
Dataset702_AtlasCS17FourGroup01
Using <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'> as reader/writer


Extracting dataset fingerprint: 100%|██████████| 615/615 [00:29<00:00, 20.87it/s]


Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': (np.int64(1536), np.int64(896)), 'median_image_size_in_voxels': array([1821., 1126.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_

Preprocessing cases: 100%|██████████| 615/615 [02:43<00:00,  3.76it/s]


Configuration: 3d_fullres...
INFO: Configuration 3d_fullres not found in plans file nnUNetPlans.json of dataset Dataset702_AtlasCS17FourGroup01. Skipping.
Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset702_AtlasCS17FourGroup01. Skipping.


CompletedProcess(args=['nnUNetv2_plan_and_preprocess', '-d', '702', '--verify_dataset_integrity'], returncode=0)

## 3. Train a single nnU-Net fold (fold 0, 250 epochs)

The model is trained as a single fold (fold 0) for 250 epochs instead of a 5-fold ensemble. If the GPU session is interrupted, the notebook automatically restores the most recently uploaded state and resumes it from `checkpoint_latest.pth`.

In [6]:
# ============================================================
# 2B. AUTOMATICALLY RESTORE THE LATEST HF TRAINING STATE FOR THIS GROUP
# ============================================================
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.utils import HfHubHTTPError

restore_dir = Path(BASE_DIR) / "restored_training_state"
try:
    snapshot_download(repo_id=HF_TRAINING_REPO_ID, repo_type="model", token=HF_TOKEN,
                       allow_patterns=[f"{RESULTS_PATH_IN_REPO}/*"],
                       local_dir=str(restore_dir))
    results_src = restore_dir / RESULTS_PATH_IN_REPO / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
    if results_src.exists():
        results_dst = Path(NNUNET_RESULTS) / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
        shutil.copytree(results_src, results_dst, dirs_exist_ok=True)
        print(f"Restored previous training state for group {GROUP_INDEX} from "
              f"{HF_TRAINING_REPO_ID}/{RESULTS_PATH_IN_REPO}")
    else:
        print(f"No previous nnUNet_results found for group {GROUP_INDEX} in restore snapshot — starting fresh.")
except HfHubHTTPError:
    print("No previous training-state repo found — starting fresh.")


Fetching 0 files: 0it [00:00, ?it/s]

No previous nnUNet_results found for group 1 in restore snapshot — starting fresh.


In [7]:
# ============================================================
# 2C. RESUME PREP
# - write checkpoint_latest.pth every 5 epochs (default is 50)
# - make sure the most advanced checkpoint is the one --c loads
# ============================================================
import importlib.util, re, torch

# (a) patch nnU-Net so checkpoint_latest.pth is saved every 5 epochs
pkg_dir = Path(importlib.util.find_spec("nnunetv2").submodule_search_locations[0])
trainer_py = pkg_dir / "training" / "nnUNetTrainer" / "nnUNetTrainer.py"
txt = trainer_py.read_text()
if "self.save_every = 50" in txt:
    trainer_py.write_text(txt.replace("self.save_every = 50", "self.save_every = 5"))
    print("Patched nnUNetTrainer: save_every = 5")
else:
    print("save_every is now:", re.findall(r"self\.save_every = \d+", txt))

# (b) after the HF restore, pick the most advanced checkpoint and expose it as checkpoint_latest.pth
fold0 = (Path(NNUNET_RESULTS) / f"Dataset{DATASET_ID:03d}_{DATASET_NAME}"
         / f"{TRAINER}__nnUNetPlans__{CONFIGURATION}" / "fold_0")
latest, best, final = (fold0 / "checkpoint_latest.pth",
                       fold0 / "checkpoint_best.pth",
                       fold0 / "checkpoint_final.pth")

def ckpt_epoch(p):
    try:
        return int(torch.load(p, map_location="cpu", weights_only=False)["current_epoch"])
    except Exception as e:
        print("Cannot read", p.name, "->", e)
        return -1

if final.exists():
    print("Training already finished (checkpoint_final.pth exists).")
elif latest.exists() or best.exists():
    e_latest = ckpt_epoch(latest) if latest.exists() else -1
    e_best = ckpt_epoch(best) if best.exists() else -1
    print(f"checkpoint_latest epoch = {e_latest}, checkpoint_best epoch = {e_best}")
    if e_best > e_latest:
        shutil.copy2(best, latest)
        print("Copied checkpoint_best.pth -> checkpoint_latest.pth")
    print("=> next cell will RESUME with --c")
else:
    print("No checkpoint found => fresh start")

Patched nnUNetTrainer: save_every = 5
No checkpoint found => fresh start


In [ ]:
# ============================================================
# 3. CHECKPOINTED + RESUMABLE SINGLE-FOLD TRAINING (fold 0, 250 epochs)
# ============================================================
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_TRAINING_REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True, private=True)

def results_dataset_dir():
    return os.path.join(NNUNET_RESULTS, f"Dataset{DATASET_ID:03d}_{DATASET_NAME}")

def fold_dir(fold):
    return os.path.join(results_dataset_dir(), f"{TRAINER}__nnUNetPlans__{CONFIGURATION}", f"fold_{fold}")

def checkpoint_state(fold):
    d = Path(fold_dir(fold))
    return {
        "exists": d.exists(),
        "latest": (d / "checkpoint_latest.pth").exists(),
        "final": (d / "checkpoint_final.pth").exists(),
    }

upload_lock = threading.Lock()

def push_results_to_hf(reason="scheduled"):
    # Copy results to a staging directory first so the HF upload sees a
    # stable snapshot rather than a directory nnU-Net is actively writing to.
    with upload_lock:
        if not os.path.isdir(NNUNET_RESULTS):
            return
        staging = Path(CHECKPOINT_CACHE) / "nnunet_results_snapshot"
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)

        source = Path(NNUNET_RESULTS)
        for p in source.rglob("*"):
            rel = p.relative_to(source)
            dst = staging / rel
            if p.is_dir():
                dst.mkdir(parents=True, exist_ok=True)
            else:
                dst.parent.mkdir(parents=True, exist_ok=True)
                try:
                    shutil.copy2(p, dst)
                except (FileNotFoundError, PermissionError, OSError):
                    # File was being written to at the exact moment of copy;
                    # the next scheduled push will pick up its final state.
                    pass

        status = {
            "upload_reason": reason,
            "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "group_index": GROUP_INDEX,
            "dataset_id": DATASET_ID,
            "dataset_name": DATASET_NAME,
            "trainer": TRAINER,
            "folds": {str(fold): checkpoint_state(fold) for fold in range(N_FOLDS)},
        }
        with open(staging / "checkpoint_status.json", "w", encoding="utf-8") as f:
            json.dump(status, f, indent=2)

        try:
            print(f"\n[HF push] uploading training state ({reason}) at {status['timestamp_utc']}...")
            api.upload_folder(
                folder_path=str(staging),
                repo_id=HF_TRAINING_REPO_ID,
                repo_type="model",
                path_in_repo=RESULTS_PATH_IN_REPO,
                commit_message=f"Checkpoint: {reason}",
            )
            print("[HF push] upload complete.")
        except Exception as e:
            print(f"[HF push] FAILED ({reason}): {e}")

def periodic_uploader(stop_event, interval_s=CHECKPOINT_UPLOAD_MINUTES * 60):
    while not stop_event.wait(interval_s):
        push_results_to_hf(reason=f"scheduled_{CHECKPOINT_UPLOAD_MINUTES}min")

stop_event = threading.Event()
uploader_thread = threading.Thread(target=periodic_uploader, args=(stop_event,), daemon=True,
                                    name="HF-checkpoint-uploader")
uploader_thread.start()
print(f"Background checkpoint uploader started: every {CHECKPOINT_UPLOAD_MINUTES} minutes.")

try:
    for fold in range(N_FOLDS):
        state = checkpoint_state(fold)
        if state["final"]:
            print(f"Fold {fold}: checkpoint_final.pth already exists. SKIPPING.")
            continue

        cmd = ["nnUNetv2_train", str(DATASET_ID), CONFIGURATION, str(fold), "-tr", TRAINER]
        if RESUME_FROM_CHECKPOINT and state["latest"]:
            cmd.append("--c")
            print(f"\nFold {fold}: RESUMING from checkpoint_latest.pth")
        else:
            print(f"\nFold {fold}: STARTING training")

        print("=" * 90); print(" ".join(cmd)); print("=" * 90)
        result = subprocess.run(cmd, text=True)
        if result.returncode != 0:
            print(f"\nFold {fold} exited with code {result.returncode}. Attempting emergency push...")
            push_results_to_hf(reason=f"fold_{fold}_failed")
            raise RuntimeError(f"Fold {fold} failed.")

        push_results_to_hf(reason=f"fold_{fold}_completed")

    print("\nAll folds completed.")
    push_results_to_hf(reason="all_folds_completed")
except Exception as e:
    print("Training failed:", e)
    push_results_to_hf(reason="emergency upload after failure")
    raise
finally:
    stop_event.set()
    uploader_thread.join(timeout=5)

print("\nTraining state permanently uploaded to:", HF_TRAINING_REPO_ID)


Background checkpoint uploader started: every 15 minutes.

Fold 0: STARTING training
nnUNetv2_train 702 2d 0 -tr nnUNetTrainer_250epochs

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-09-30 23:45:09.065065: Using torch.compile...
2026-09-30 23:45:11.518226: do_dummy_2d_data_aug: False
2026-09-30 23:45:11.519884: Creating new 5-fold cros

W0930 23:45:36.563000 29312 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode



[HF push] uploading training state (scheduled_15min) at 2026-09-30T23:59:57Z...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

[HF push] upload complete.

[HF push] uploading training state (scheduled_15min) at 2026-10-01T06:50:58Z...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[HF push] upload complete.
 Epoch 177
2026-10-01 06:28:51.338737: Current learning rate: 0.0033
2026-10-01 06:31:04.953118: train_loss -0.2976
2026-10-01 06:31:04.953525: val_loss -0.2791
2026-10-01 06:31:04.953867: Pseudo dice [np.float32(0.9255), np.float32(0.9331), np.float32(0.9173), np.float32(0.8698), np.float32(0.9109), np.float32(0.9351), np.float32(0.9768), np.float32(0.9563), np.float32(0.9358), np.float32(0.8952), np.float32(0.8466), np.float32(0.8944), np.float32(0.7974), np.float32(0.8056), np.float32(0.9316), np.float32(0.8904), np.float32(0.9851), np.float32(0.941), np.float32(0.8548), np.float32(0.8896), np.float32(0.7721), np.float32(0.9684)]
2026-10-01 06:31:04.954092: Epoch time: 133.62 s
2026-10-01 06:31:04.954423: Yayy! New best EMA pseudo Dice: 0.8985999822616577
2026-10-01 06:31:07.174981: 
2026-10-01 06:31:07.175402: Epoch 178
2026-10-01 06:31:07.175629: Current learning rate: 0.00326
2026-10-01 06:33:20.863931: train_loss -0.3009
2026-10-01 06:33:20.864181: val

In [ ]:
# ============================================================
# OPTIONAL -- only run if training crashed with "No space left on device".
# EMERGENCY: free disk space, then push final training state to HF
# (run this BEFORE factory-resetting the Kaggle session)
# ============================================================
import shutil
from pathlib import Path

print(shutil.disk_usage("/"))  # total, used, free (bytes)

# Delete the partially-written / unneeded validation probability exports
# (the .npz/.pkl files that can trigger "No space left on device"). These
# are optional softmax exports from the validation step, not the trained
# model itself, and are safe to delete.
val_dir = Path(fold_dir(0)) / "validation"
if val_dir.exists():
    junk_files = list(val_dir.rglob("*.npz")) + list(val_dir.rglob("*.pkl"))
    freed = sum(f.stat().st_size for f in junk_files if f.exists())
    for f in junk_files:
        try:
            f.unlink()
        except Exception as e:
            print("could not delete", f, e)
    print(f"Deleted {len(junk_files)} files, freed ~{freed/1e9:.2f} GB")

print("Disk after cleanup:", shutil.disk_usage("/"))

print("\n[HF push] uploading final training state (pre-reset)...")
api.upload_folder(
    folder_path=NNUNET_RESULTS,
    repo_id=HF_TRAINING_REPO_ID,
    repo_type="model",
    path_in_repo=RESULTS_PATH_IN_REPO,
    commit_message="Checkpoint: pre-factory-reset final push",
    ignore_patterns=["*.npz"],  # skip bulky optional probability exports
)
print("[HF push] upload complete. Safe to factory reset now.")


## 4. Held-out slice evaluation for this group — the headline metric

This is now the primary evidence of segmentation quality for **this group's model**. It uses
the actual final checkpoint + full nnU-Net inference pipeline on the held-out slices defined by
this group's fixed-seed split from the data-prep notebook.

**Why these slices are genuinely unseen, not just nominally "held out":** in an earlier version
of this pipeline, all slices (including the ones meant to be held out) were exported into
`imagesTr`/`labelsTr`, and only a side manifest recorded which ones were "supposed" to be held
out. nnU-Net doesn't read that file — its own 5-fold split is built from everything physically
present in `imagesTr`, so those slices were actually trained on in 4 of 5 folds. That's fixed
now: held-out slices live in a separate `heldout/` folder that was never copied into, scanned
by, or referenced during `nnUNetv2_plan_and_preprocess` or `nnUNetv2_train` at any point.
Section 1B's leakage-guard assertion checks this on disk every run, not just once.

**Scope note (read before citing this number anywhere):** these are held-out *slices from one
specimen* (CS17), not a held-out *specimen*. Class selection into groups, preprocessing
decisions, and the overall imaging/staining domain are shared between train and held-out sets
here. This is a legitimate check against overfitting to specific slices, but it does not by
itself demonstrate that the model generalizes to a different specimen or a different imaging
domain, and it says nothing about how this group's model performs on another group's
structures. There is currently no cross-specimen validation set in this pipeline.


In [ ]:
# ============================================================
# 4. FINAL TRAINING OUTPUT AUDIT
# ============================================================
print("Results directory:", results_dataset_dir())

for fold in range(N_FOLDS):
    state = checkpoint_state(fold)
    print(f"Fold {fold}: exists={state['exists']}, latest={state['latest']}, final={state['final']}")


In [ ]:
# ============================================================
# 4B. HELD-OUT SLICE EVALUATION (fixed-seed split from notebook 1)
# ============================================================
holdout_cases = split_manifest["holdout_cases"]
print(f"Evaluating on {len(holdout_cases)} held-out slices (never used for gradient updates in any fold)")

holdout_input_dir = os.path.join(BASE_DIR, "holdout_imagesTs")
holdout_gt_dir = os.path.join(BASE_DIR, "holdout_labelsTs")
holdout_pred_dir = os.path.join(BASE_DIR, "holdout_predictions")
for d in [holdout_input_dir, holdout_gt_dir, holdout_pred_dir]:
    os.makedirs(d, exist_ok=True)

for case_id in holdout_cases:
    shutil.copy2(os.path.join(heldout_images_dir, f"{case_id}_0000.png"),
                 os.path.join(holdout_input_dir, f"{case_id}_0000.png"))
    shutil.copy2(os.path.join(heldout_labels_dir, f"{case_id}.png"),
                 os.path.join(holdout_gt_dir, f"{case_id}.png"))

missing = [fold for fold in range(N_FOLDS) if not checkpoint_state(fold)["final"]]
if missing:
    raise RuntimeError(f"Cannot run held-out inference. Missing final checkpoints for folds: {missing}")

subprocess.run([
    "nnUNetv2_predict",
    "-i", holdout_input_dir, "-o", holdout_pred_dir,
    "-d", str(DATASET_ID), "-c", CONFIGURATION,
    "-tr", TRAINER,
    "-f", *[str(f) for f in range(N_FOLDS)],
], check=True)
print("Held-out inference complete:", holdout_pred_dir)


In [ ]:
# ============================================================
# 4C. SCORE HELD-OUT PREDICTIONS (per-class Dice)
# ============================================================
import numpy as np
from PIL import Image
import pandas as pd

def dice(pred, gt, class_id):
    p = (pred == class_id)
    g = (gt == class_id)
    if g.sum() == 0 and p.sum() == 0:
        return None
    intersection = np.logical_and(p, g).sum()
    return 2.0 * intersection / (p.sum() + g.sum() + 1e-8)

nnunet_id_to_name = {int(v): k for k, v in dataset_json["labels"].items()}

rows = []
for case_id in holdout_cases:
    pred = np.array(Image.open(os.path.join(holdout_pred_dir, f"{case_id}.png")))
    gt = np.array(Image.open(os.path.join(holdout_gt_dir, f"{case_id}.png")))
    for class_id, class_name in nnunet_id_to_name.items():
        if class_id == 0:
            continue
        d = dice(pred, gt, class_id)
        if d is not None:
            rows.append({"case_id": case_id, "class_id": class_id, "class_name": class_name, "dice": d})

dice_df = pd.DataFrame(rows)
per_class_summary = dice_df.groupby("class_name")["dice"].agg(["mean", "median", "std", "count"]).sort_values("mean")
print("Per-class held-out Dice (held-out SLICES from one specimen — see scope note above):")
print(per_class_summary.to_string())
print(f"\nOverall mean Dice across all classes/slices: {dice_df['dice'].mean():.3f}")

dice_df.to_csv(os.path.join(BASE_DIR, "holdout_dice_scores.csv"), index=False)
push_results_to_hf(reason="held-out evaluation scores")


In [ ]:
# ============================================================
# 4D. VISUALIZE PREDICTIONS VS GROUND TRUTH (held-out slices)
# Transparent colored overlay on top of the real histology image
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import numpy as np

# One fixed RGB color per class id (index 0 reserved for background, unused in overlay)
class_ids_sorted = sorted(nnunet_id_to_name.keys())
cmap_base = plt.get_cmap("tab20", max(len(class_ids_sorted) - 1, 1))
class_colors = {0: (0, 0, 0)}  # background — never drawn
for i, cid in enumerate([c for c in class_ids_sorted if c != 0]):
    class_colors[cid] = cmap_base(i)[:3]  # RGB, drop alpha

ALPHA = 0.45  # overlay transparency — raise/lower to taste

def make_overlay(gray_img, label_map, alpha=ALPHA):
    """Blend a colored label map onto the grayscale image, background left untouched."""
    base = np.stack([gray_img.astype(np.float32) / 255.0] * 3, axis=-1)
    overlay = base.copy()
    for cid, color in class_colors.items():
        if cid == 0:
            continue
        mask = label_map == cid
        if not mask.any():
            continue
        color_arr = np.array(color).reshape(1, 1, 3)
        overlay[mask] = (1 - alpha) * base[mask] + alpha * color_arr
    return overlay

def show_pred_vs_gt(case_id):
    img  = np.array(Image.open(os.path.join(holdout_input_dir, f"{case_id}_0000.png")))
    pred = np.array(Image.open(os.path.join(holdout_pred_dir, f"{case_id}.png")))
    gt   = np.array(Image.open(os.path.join(holdout_gt_dir, f"{case_id}.png")))

    gt_overlay = make_overlay(img, gt)
    pred_overlay = make_overlay(img, pred)

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    axes[0].imshow(img, cmap="gray")
    axes[0].set_title(f"{case_id} — Histology (input)")

    axes[1].imshow(gt_overlay)
    axes[1].set_title("Ground truth (overlay)")

    axes[2].imshow(pred_overlay)
    axes[2].set_title("Prediction (overlay)")

    for ax in axes:
        ax.axis("off")

    present = sorted(set(np.unique(gt)) | set(np.unique(pred)))
    present = [c for c in present if c != 0]
    handles = [mpatches.Patch(color=class_colors[c], label=nnunet_id_to_name[c])
               for c in present if c in nnunet_id_to_name]
    if handles:
        fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 6),
                    bbox_to_anchor=(0.5, -0.05), fontsize=9)

    plt.tight_layout()
    plt.show()

# Show a handful of held-out cases. Change N_EXAMPLES or pass specific case_ids.
N_EXAMPLES = 5
for case_id in holdout_cases[:N_EXAMPLES]:
    show_pred_vs_gt(case_id)

In [ ]:
# ============================================================
# PREVIEW: held-out evaluation with checkpoint_best.pth (model NOT finished)
# Separate from the final evaluation cells (4B/4C/4D). Do not cite these numbers as final.
# ============================================================
import numpy as np, pandas as pd
from PIL import Image

prev_in   = os.path.join(BASE_DIR, "preview_imagesTs")
prev_gt   = os.path.join(BASE_DIR, "preview_labelsTs")
prev_pred = os.path.join(BASE_DIR, "preview_predictions")
for d in [prev_in, prev_gt, prev_pred]:
    os.makedirs(d, exist_ok=True)

holdout_cases = split_manifest["holdout_cases"]
for case_id in holdout_cases:
    shutil.copy2(os.path.join(heldout_images_dir, f"{case_id}_0000.png"),
                 os.path.join(prev_in, f"{case_id}_0000.png"))
    shutil.copy2(os.path.join(heldout_labels_dir, f"{case_id}.png"),
                 os.path.join(prev_gt, f"{case_id}.png"))

subprocess.run([
    "nnUNetv2_predict",
    "-i", prev_in, "-o", prev_pred,
    "-d", str(DATASET_ID), "-c", CONFIGURATION,
    "-tr", TRAINER,
    "-f", "0",
    "-chk", "checkpoint_best.pth",
], check=True)

def dice(pred, gt, class_id):
    p, g = (pred == class_id), (gt == class_id)
    if g.sum() == 0 and p.sum() == 0:
        return None
    return 2.0 * np.logical_and(p, g).sum() / (p.sum() + g.sum() + 1e-8)

id_to_name = {int(v): k for k, v in dataset_json["labels"].items()}
rows = []
for case_id in holdout_cases:
    pred = np.array(Image.open(os.path.join(prev_pred, f"{case_id}.png")))
    gt   = np.array(Image.open(os.path.join(prev_gt, f"{case_id}.png")))
    for cid, cname in id_to_name.items():
        if cid == 0:
            continue
        d = dice(pred, gt, cid)
        if d is not None:
            rows.append({"case_id": case_id, "class_name": cname, "dice": d})

prev_df = pd.DataFrame(rows)
summary = prev_df.groupby("class_name")["dice"].agg(["mean", "median", "std", "count"]).sort_values("mean")
print("PREVIEW per-class held-out Dice (checkpoint_best, unfinished model):")
print(summary.to_string())
print(f"\nPREVIEW overall mean Dice: {prev_df['dice'].mean():.3f}")
prev_df.to_csv(os.path.join(BASE_DIR, "preview_dice_scores.csv"), index=False)

## Checkpoint preservation policy

- The notebook uploads the complete available `nnUNet_results` state for **this GROUP_INDEX only** to
  `senku111/atlas-cs17-4groups-nnunet-training` (a **model**-type HF repo), under that group's own
  folder (`RESULTS_PATH_IN_REPO`), every **30 minutes**
  and immediately after each completed fold.
- Uploads are staged to a local cache directory first, file-by-file, so a checkpoint that
  nnU-Net is actively writing to at that instant is skipped rather than uploaded corrupted
  — the next scheduled push picks it up once it's finished writing.
- If training fails, it attempts an emergency upload before stopping.
- `CLEAN_RESULTS` is a hard-disabled safety guard: it must stay `False`, and the notebook
  raises immediately if it's ever set `True`. Delete results manually only if you
  intentionally want to wipe a run.
- The emergency pre-factory-reset cell deletes bulky optional validation `.npz`/`.pkl`
  probability exports before pushing, since these have previously caused
  "No space left on device" on Kaggle.
- The latest `checkpoint_latest.pth`, completed `checkpoint_final.pth`, logs, progress
  files, and other nnU-Net result artifacts are preserved.
- After a new Kaggle session, the restore section downloads the latest uploaded state and
  resumes any incomplete folds.
- Held-out evaluation scores are pushed alongside the checkpoints so the reported Dice
  numbers travel with the exact checkpoint that produced them.
- Training uses `nnUNetTrainer_250epochs` on a single fold (fold 0 only, 250 total epochs, no ensemble).


## Important scientific expectation

This is a **per-group** model: it only ever sees, predicts, and is scored on the structure IDs assigned to `GROUP_INDEX`. It says nothing about the other groups' structures — those are trained (and evaluated) by separate runs of this same notebook with a different `GROUP_INDEX`.

This notebook trains and evaluates a single nnU-Net fold (250 epochs, no ensemble) on slices from **one Atlas specimen (CS17-6521)**. The held-out evaluation in Section 4 is a legitimate, standard check
against slice-level overfitting: those slices never contributed to any fold's gradient
updates.

**What this does and does not demonstrate:** because the train and held-out sets are drawn
from the same specimen — same imaging domain, same staining, same acquisition — a strong
held-out Dice score shows the model is not simply memorizing individual training slices. It
does **not** show that the model generalizes to a different specimen, a different imaging
modality, or a different staining protocol. There is no specimen held out of this evaluation,
because there is currently only one specimen with expert ground-truth labels in this pipeline.

A prior version of this notebook ran the trained model on the Virtual Human Embryo (VHE)
Stage 15 / Carnegie #721 archive and called that an "external validation" step. It has been
removed: VHE has no expert ground-truth segmentation, so there was no way to score whether
those predictions were actually correct — only whether they looked plausible. A plausible-
looking but unscored prediction is not evidence of generalization, and presenting it as such
overstated what the pipeline had actually verified.

Genuine cross-specimen validation, if wanted later, would require a second specimen with its
own expert-labeled ground truth (or a small fine-tuning set), which is a distinct future
project rather than something bolted onto this training notebook.